# A First Naïve Model

> Scotland’s nature is ...
> 
> Nature gives us ...
> 
> ... important nature and landscape interests ...
>
> ... connect with nature and create moments ...


```mermaid
flowchart LR

st("[START]") -- 50% --> sc(scotlands)
sc -- 100% --> na(nature)
na -- 25% --> is(is)
is --> dde1("`_..._`")

st -- 50% --> na(nature)
na -- 25% --> gi(gives)
gi -- 100% --> us(us)
us --> dde2("`_..._`")


dds1("`_..._`") --> im(important)
im -- 100% --> na(nature)
na -- "`**50%**`" --> an(and)
an --> la(landscape)
la -- 100% --> in(interests)
in --> dde3("`_..._`")

dds2("`_..._`") --> co(connect)
co -- 100% --> wi(with)
wi -- 100% --> na
an --> cr(creates)
cr -- 100% --> mo(moments)
mo --> dde4("`_..._`")
```

In [1]:
import pickle
from itertools import accumulate
from random import randint
import ipywidgets as widgets
from IPython.display import display, Markdown


def upsert(into_model, from_token, to_token):

    # Not seen our "from" before? Create the entry.
    if from_token not in into_model:
        into_model[from_token] = {to_token: 1}
        return

    # Seen the "from", but not the "to"? Add this as an option.
    if to_token not in into_model[from_token]:
        into_model[from_token][to_token] = 1
    # Seen the "from" and the "to"? Add weight to this option.
    else:
        into_model[from_token][to_token] += 1

In [2]:
sentences_words = pickle.load(open('sentences_words.pkl', 'rb'))


def build_forward1_model(sentences_words):
    model = {}
    # Loop over every sentence in every blog.
    for sentence in sentences_words:
        tokens = ['START'] + sentence + ['END']

        # For each word "from" & "to" pair, add them to the model.
        for idx in range(0, len(tokens) - 1):
            token_prev = tokens[idx]
            token_this = tokens[idx + 1]
            upsert(model, token_prev, token_this)

    # Save the model for re-use.
    return model


# Build the model: for every "from" word, how often does each "to" word follow it?
# (uses the upsert function defined above)
model = build_forward1_model(sentences_words)

# Save the model for re-use.
pickle.dump(model, open('models/forward_1.pkl', 'wb'))

In [3]:
def next_word(in_model, this_word):
    # Find all the "to" words for our "from".
    candidates = in_model[this_word].items()
    words = [v[0] for v in candidates]
    scores = [v[1] for v in candidates]

    # Score each "to" word based on its weight, and line it up.
    # e.g 1 = A, 2-6 = B, 7-9 = C, etc.
    upper_lim = sum(scores)
    accumulated_scores = accumulate(scores)
    choices = zip(accumulated_scores, words, strict=True)

    # Roll a die based on the maximum number.
    die_roll = randint(0, upper_lim)

    # Pick the one that matches our dice roll.
    chosen = list(filter(lambda c: c[0] >= die_roll, choices))[0][1]
    return chosen


# Pick a likely "first" word for a sentence.
word = widgets.Output(layout={'padding': '16px'})
with word:
    display(Markdown('### ' + next_word(model, 'START')))

word

Output(layout=Layout(padding='16px'))

In [4]:
def generate_sentence(in_model):
    # Generate sentences until a natural end, with a cap of 55 words.
    last_word = 'START'
    sentence = []
    for _ in range(55):
        word = next_word(in_model, last_word)
        last_word = word
        if last_word == 'END':
            break
        sentence.append(word)

    return sentence


# Generate a random sentence.
generated_sentence = ' '.join(generate_sentence(model)) + '.'
out = widgets.Output(layout={'padding': '16px'})
with out:
    display(Markdown('### ' + generated_sentence.capitalize()))

out

Output(layout=Layout(padding='16px'))